###Working with Nulls
----------------------------------------------------------------------------------------------------------------------------------------------

####Handling Null values
 1. Equality check
 2. Null in expressions
 3. [Conditional functions for Null](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#conditional-functions)
 4. Null in aggregations

####1. Create a data frame to demo some scenarios

In [0]:
person_list = [(100, "Prashant", 30),
             (101, "David", None),
             (102, "Sushant",None),
             (103, "Abdul", 45),
             (104, "Shruti", 28)]
             
person_df = spark.createDataFrame(person_list).toDF("id", "name", "age")
display(person_df)

####2. How null equality is executed.

2.1 Find all records where age is 28.\
2.1 Find all records where age is not given or unknown.

In [0]:
person_df.where("age == 28").display()

2.2 Create a boolean column to investigate

In [0]:
from pyspark.sql.functions import expr

person_df.withColumn("age_null_expr", expr("age IS null")).display()

2.3 Use case for null equality

 Select only those persons having a valid age information

In [0]:
person_df.where("age IS null").display()

####3. How operators work on null values

3.1 Check the result of > operator on null values

In [0]:
person_df.withColumn("age_gt_29", expr("age > 29")).display()

3.2 How the comperison operator behaves in answering business questions.

 Find all employees where age is greater than 29

In [0]:
person_df.where("age > 29").display()

3.3 How mathametical operators work on null.\
 Calculate experience for every employee using the following formula.\
 experience = age - 23

In [0]:
person_df.withColumn("experience", expr("age - 23")).display()

3.4. Calculate the experience knowing that age could be null.\
 If age is null then assume 23 years for experience calculation.

In [0]:
person_df.withColumn("experience", expr("nvl(age,23) - 23")).display()

####4. How aggregates work on null values

4.1 What is the average age?

In [0]:
person_df.selectExpr("avg(age)").display()

4.2 What if we filter out null values before aggregation

In [0]:
person_df.where("age is not null").selectExpr("avg(age)").display()

###Working with Numbers
----------------------------------------------------------------------------------------------------------------------------------------------

####Working with numbers
 1. Using mathematical expressions
 2. [Using mathematical functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#mathematical-functions)
 3. [Using aggregate functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#aggregate-functions)

####1. Load invoices data to a dataframe

In [0]:
retail_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/invoices.csv")
)

display(retail_df)

####2. Calculate total_value for each invoice line item

2.1 Simple approach of creating expressions

In [0]:
from pyspark.sql.functions import expr
retail_df.withColumn("total_value", expr("round(quantity * unitprice,2)")).display()

2.2 Using column expressions

In [0]:
from pyspark.sql.functions import col, round

retail_df.withColumn("total_value", round(col("quantity") * col("unitprice"), 2)).display()

2.3 Using expression variable - 1

In [0]:
total_value_expr1 = round(col("quantity") * col("unitprice"), 2)

retail_df.withColumn("total_value", total_value_expr1).display()

2.4 Using expression variable - 2

In [0]:
total_value_expr2 = expr("round(quantity * unitprice, 2)")

retail_df.withColumn("total_value", total_value_expr2).display()

####3. Perform the following exploratory analysis on invoices data
 1. Can we make invoice numbers a numeric field?
 2. Analyize quantity to identify potentially invalid records
 3. Analyze unit price to identify potentially invalid records

3.1 Analyze using dataframe summary

 Available statistics are:
 ```
   count - mean - stddev - min - max - approximate percentiles
 ```

In [0]:
#retail_df.describe(['InvoiceNo', 'Quantity', 'UnitPrice']).display()
retail_df.summary().select('summary', 'InvoiceNo', 'Quantity', 'UnitPrice').display()

3.2 Using sql functions

In [0]:
from pyspark.sql.functions import min, max, percentile

retail_df.select(min("unitprice"),max("unitprice"), percentile("unitprice",0.99)).display()

###Manipulating Strings
----------------------------------------------------------------------------------------------------------------------------------------------

####Working with String
 1. [String Manipulation Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#string-functions)

####1. Requirement
 You are given a dataframe below
 ```
   df = spark.createDataFrame([("25","07","1982")]).toDF("day", "month", "year")
 ```
 Create a date from day, month, and year.

In [0]:
from pyspark.sql.functions import to_date,concat_ws #concat with separator

df = spark.createDataFrame([("25","07","1982")]).toDF("day", "month", "year")
#df.display()

date_expr = to_date(concat_ws("-","year","month","day")).alias("Date") 
df.select(date_expr).display()


####2. Requirement
 You are given a dataframe below
 ```
   df = (spark.createDataFrame([("Sanjay", "Kalra", 25, "July", 1982, 19408.98)])
           .toDF("first_name", "last_name", "day", "month", "year", "salary"))
 ```
 Create the following output
 ```
   +-----------------------------------------------+----------+
   |fun_text                                       |salary    |
   +-----------------------------------------------+----------+
   |Mr. Sanjay Kalra was born on 25th July of 1982.|$19,408.98|
   +-----------------------------------------------+----------+
 ```

In [0]:
from pyspark.sql.functions import format_number, format_string

df = (spark.createDataFrame([("Sanjay", "Kalra", 25, "July", 1982, 19408.98)])
          .toDF("first_name", "last_name", "day", "month", "year", "salary"))

salary_fmt = format_number("salary","$###,###.##").alias("salary")
text_fmt = format_string("Mr. %s %s was born on %dth %s of %d.","first_name","last_name","day","month","year").alias("fun_text")

df.select(text_fmt, salary_fmt).display()


####3. Requirement
 You are given a dataframe below
 ```
   df = spark.createDataFrame([("Benga`li Market", "110001"),("Adu~godi", "560030")]) \
           .toDF("address", "pin")
 ```
 Write code to fix the data problem in the address field

In [0]:
from pyspark.sql.functions import translate #translate function replaces all occurrences of the character in the first argument with the character in the second argument

df = spark.createDataFrame([("Benga`li Market", "110001"),("Adu~godi", "560030")]) \
          .toDF("address", "pin")
df.withColumn("address", translate("address","~`","")).display()

###Working with Date
----------------------------------------------------------------------------------------------------------------------------------------------

####Working with dates
 1. [Date functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. Requirement
 You are given a dataframe as below

In [0]:
from pyspark.sql.functions import concat_ws

data_list = [(2022, 5, 18) , (9999, 12, 31), (-9999, 1, 1), (10000, 1, 1), 
             (-10000, 1, 1), (193, 5, 25),   (99, 5, 25),   (1000, 2, 29)]

df = (spark.createDataFrame(data_list).toDF("Y", "M", "D")
     .withColumn("date_str", concat_ws("-", "Y", "M", "D")))

df.display()

####2. Convert strings to date

 1. Spark validates the date against the Proleptic Gregorian calendar.
 2. The negative years are BC, and the positive values are AD in Gregorian calander.
 3. Valid dates are taken, and invalid dates throw an exception or taken as null

In [0]:
from pyspark.sql.functions import expr
df = (
    df.withColumn("valid_date", expr("try_to_date(date_str, 'y-M-d')"))
    .drop("D","M","Y")
)
df.display()

####3. Add, Subtract days and months to date

In [0]:
from pyspark.sql.functions import date_add, date_sub, add_months

df = (
    df.withColumns({
        "add_5_days": date_add("valid_date",5),
        "sub_5_days": date_sub("valid_date",5),
        "add_5_months": add_months("valid_date",5),
        "sub_5_months": add_months("valid_date",-5)
    
    })
)

df.display()

####4. Current date, date difference, and interval

In [0]:
from pyspark.sql.functions import current_date, date_diff, col
df = (
    df.withColumns({
        "current_date": current_date(),
        "delta_date_days": date_diff("add_5_days","valid_date"),
        "delta_date_interval": col("current_date")-col("valid_date")
    })
)

df.display()

####5. Format date

In [0]:
from pyspark.sql.functions import date_format #date_format returns a string datatype not a date datatype

df = (
    df.withColumn("fmt_date", date_format("valid_date", "dd MMM yyyy"))
)

df.display()

###Working with Timestamps
----------------------------------------------------------------------------------------------------------------------------------------------

####Working with timestamp
 1. [Timestamp functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. How Spark stores the timestamp

 1. Timestamp is internally stored as 12 byte integer known as INT96
 2. Timestamp is made up of 7 fields
     1. Year
     2. Month
     3. Day
     4. Hour
     5. Minute
     6. Second
         * Up to 6 decimal places
         * Microsecond precision
     7. Timezone

####2. Requirement
 You are given the below dataframes

In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

2.1 covert the df_1 to timestamp

In [0]:
from pyspark.sql.functions import to_timestamp
df_1.select("string_time",
             to_timestamp("string_time", "yyyy-MM-dd'T'HH:mm:ss.SSSS").alias("valid_time")
             ).display()

2.2 Convert the df_2 to time

In [0]:
df_2.selectExpr(
    "string_time",
    "to_timestamp(string_time, 'dd-MM-yyyy HH:mm:ss.SSSS') as valid_time"
).display()

2.3 Convert the df_3 to time

In [0]:
df_3.selectExpr(
    "string_time",
    "try_to_timestamp(string_time, 'yyyy-MM-dd HH:mm:ss.SSSS') as valid_time"
).display()

###3. Timezone information

 1. A timestamp without timezone information is incomplete.
 2. Spark offers two data types for timestamp
     1. TIMESTAMP
     2. TIMESTAMP_NTZ
 3. For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
 4. Session timezone is specified as spark.sql.session.timeZone

3.1 What is your default session timezone?

In [0]:
spark.conf.get("spark.sql.session.timeZone")

3.2 Change your session timezone to IST

In [0]:
spark.conf.set("spark.sql.session.timeZone","Etc/UTC")

In [0]:
spark.conf.get("spark.sql.session.timeZone")

####4. Working with NTZ data

4.1 Load machine-events-no-tz.csv file and show the data

In [0]:
event_ntz_schema = "component string, event_time string, reading string"

event_ntz_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(event_ntz_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv")
)

event_ntz_df.display()

4.2 Parse te event_time to a TIMESTAMP_NTZ value

In [0]:
from pyspark.sql.functions import to_timestamp_ntz, lit

event_valid_ntz_df = (
    event_ntz_df.withColumn("event_time_valid_ntz", to_timestamp_ntz("event_time", lit("dd-MM-yyyy HH:mm:ss.SSS")))
)

event_valid_ntz_df.display()

4.3 event_time_ntz field to a valid timestamp value\
 Assume the event_time_ntz is IST time

In [0]:
from pyspark.sql.functions import convert_timezone, lit

stz = spark.conf.get("spark.sql.session.timeZone")

events_df = (
    event_valid_ntz_df.withColumn("event_time_tz", to_timestamp(convert_timezone(lit("IST"), lit(stz), "event_time_valid_ntz")))
)

events_df.display()

####5. Working with TZ data

5.1 Load machine-events-with-tz.csv file and show the data

In [0]:
event_tz_schema = "component string, event_time_tz_str string, reading string"

events_tz_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(event_tz_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-with-tz.csv")
)

display(events_tz_df)

5.2 Parse the event_time field to a valid timestamp value\
 Timezone information is provided in the data file

In [0]:
from pyspark.sql.functions import to_timestamp
event_data_df = (
    events_tz_df.withColumn("event_time_tz", to_timestamp("event_time_tz_str", "dd-MM-yyyy HH:mm:ss.SSSZ"))
)
event_data_df.display()

###Working with Complex Data Types
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Requirement
 Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

####2. Analysis Requirement
 We want to know country wise student count.

In [0]:
%sql
with offline_students(
select id, from_json(address,
    """struct<AddressLine1 string,
       AddressLine2 string,
       City string,
       Country string,
       Pin string,
       State string>
    """ ) as address
from dev.spark_db.offline_students_raw
)
select address.country, count(*) as head_count from offline_students group by address.country


####3. Requirement
 Prepare an offline_students table which is ready for analysis

 Complex Data Types in Spark
 1. Struct
 2. Array
 3. Map

In [0]:
from pyspark.sql.functions import from_json

#{""AddressLine1"":""D104 Gopalan Squire"",""AddressLine2"":""Whitefield"",""City"":""Bangalore"",""Country"":""India"",""Pin"":""560001"",""State"":""Karnataka""}
address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"

#[{""Skill"":""Apache Spark"",""YearsOfExperience"":""5""},{""Skill"":""Apache Kafka"",""YearsOfExperience"":""6""}] ----it has square brackets in start and end which suggest its an array
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"

#{""email"":""xyz@abc.com"",""phone"":""9823128923""} {""phone"":""6984753281"",""whatsapp"":""6924587322""}-- there is no consistency in field name thats why it can be struct. we dont define any field its key value pair which is represented by string, string
contacts_schema = "map<string,string>"


offline_student_df = (
    offline_students_raw_df.withColumns({
        "Address": from_json("Address", address_schema),
        "Skills": from_json("SKills", skills_schema),
        "Contacts": from_json("Contacts", contacts_schema)
    })
)

#offline_student_df.display()
offline_student_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students")

####4. Requirement
 Perform the following analysis
 1. What is country wise student count.
 2. Find all students with more than 1 years of Spark knowledge
 3. Find all students who didn't provide phone or whatsapp

4.1 What is country wise student count.

In [0]:
%sql
select Address.country, count(*) as count from dev.spark_db.offline_students group by Address.country

4.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql
with offline_student_skills (select id, FirstName, LastName, explode(Skills) as skills from dev.spark_db.offline_students)
select id, FirstName, LastName, skills.* from offline_student_skills where skills.Skill like '%Spark%' and skills.YearsOfExperience > 1

4.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql
select * from dev.spark_db.offline_students where contacts['phone'] is null and contacts['whatsapp'] is null

###Working with JSON data
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Requirement
 Read data from students_online.json file and load into online_students table.

In [0]:
online_student_schema = """
    ID string, FirstName string, LastName string,
    Address struct<AddressLine1 string, AddressLine2 string, City string, State string, Country string, Pin string>,
    Skills array<struct<Skill string, YearsOfExperience string>>,
    Contacts map<string, string>
    """

online_students_df = (
    spark.read.format("json")
    .schema(online_student_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_online.json")
)

#online_students_df.display()
online_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.online_students")

####2. Requirement
 Perform the following analysis
 1. What is country wise student count.
 2. Find all students with more than 1 years of Spark knowledge
 3. Find all students who didn't provide phone or whatsapp

2.1 What is country wise student count.

In [0]:
%sql

 select Address.Country, count(*) as count
 from dev.spark_db.online_students
 group by Address.Country

2.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

 with online_students_skills(
   select id, firstname, LastName, explode(Skills) as skills
   from dev.spark_db.online_students)
 select id, firstname, lastname, skills.*
 from online_students_skills
 where skills.skill like "%Spark%" and skills.YearsOfExperience > 1

2.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

 select id, FirstName, LastName, Contacts['email'] as email
 from dev.spark_db.online_students
 where Contacts['phone'] is null and Contacts['whatsapp'] is null

###Working with Variant Type
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Requirement
 Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "id string, first_name string, last_name string, address string, skills string, contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.spark_db.offline_students_raw")

####2. Requirement
 Prepare an offline_var_students table which is ready for analysis

In [0]:
from pyspark.sql.functions import parse_json 
#parse_json no need to define schema as we do in from_json. Parse_json will figure out the schema and transform it into a variant data type.

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": parse_json("address"),
        "skills": parse_json("skills"),
        "contacts": parse_json("contacts")
    })
)
#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_var_students")

####3. Requirement
 Perform the following analysis
 1. What is country wise student count.
 2. Find all students with more than 1 years of Spark knowledge
 3. Find all students who didn't provide phone or whatsapp

2.1 What is country wise student count.

In [0]:
%sql
 -- Variant object element names are case sensitive

 select cast(address:Country as string), count(*) as count
 from dev.spark_db.offline_var_students
 group by cast(address:Country as string)

2.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql
with offline_students_skills(
select id , first_name, last_name, cast(value:Skill as string), cast(value:YearsOfExperience as int)  --value is column returned by variant_explode
from dev.spark_db.offline_var_students, lateral variant_explode_outer(skills)  -- lateral allows a subquery or function (like explode) to use columns from the current row of the main table. Variant_explode_outer will show the record even if the skill is null
)
select * from offline_students_skills where skill like "%Spark%" and yearsofexperience>1

2.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

 select id, first_name, last_name, contacts:email
 from dev.spark_db.offline_var_students
 where contacts:phone is null and contacts:whatsapp is null